In [93]:
import pandas as pd

In [94]:

df = pd.read_csv(
    'Amazon_Reviews.csv',
    engine='python', #fix memory/buffer issues 
)
df.head(5)

,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience
0,Eugene ath,/users/66e8185ff1598352d6b3701a,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...","September 16, 2024"
1,Daniel ohalloran,/users/5d75e460200c1f6a6373648c,GB,9 reviews,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...,"September 16, 2024"
2,p fisher,/users/546cfcf1000064000197b88f,GB,90 reviews,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,"September 16, 2024"
3,Greg Dunn,/users/62c35cdbacc0ea0012ccaffa,AU,5 reviews,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,"September 17, 2024"
4,Sheila Hannah,/users/5ddbe429478d88251550610e,GB,8 reviews,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,"September 16, 2024"


In [95]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (21214, 9)
Columns: ['Reviewer Name', 'Profile Link', 'Country', 'Review Count', 'Review Date', 'Rating', 'Review Title', 'Review Text', 'Date of Experience']


In [96]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21214 entries, 0 to 21213
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Reviewer Name       21214 non-null  object
 1   Profile Link        21163 non-null  object
 2   Country             21054 non-null  object
 3   Review Count        21055 non-null  object
 4   Review Date         21055 non-null  object
 5   Rating              21055 non-null  object
 6   Review Title        21055 non-null  object
 7   Review Text         21055 non-null  object
 8   Date of Experience  20947 non-null  object
dtypes: object(9)
memory usage: 1.5+ MB


In [97]:
df.isnull().sum()

Reviewer Name           0
Profile Link           51
Country               160
Review Count          159
Review Date           159
Rating                159
Review Title          159
Review Text           159
Date of Experience    267
dtype: int64

In [98]:
# Clean column names
df.columns = [col.strip().replace(' ', '_').lower() for col in df.columns]
print("Columns:", df.columns.tolist())

Columns: ['reviewer_name', 'profile_link', 'country', 'review_count', 'review_date', 'rating', 'review_title', 'review_text', 'date_of_experience']


In [99]:
# Drop exact duplicate rows (keep first)
df = df.drop_duplicates().reset_index(drop=True)

In [100]:
#remove whitespace from reviewer_name and convert to string
df['reviewer_name'] = df['reviewer_name'].astype(str).str.strip()

In [101]:
#remove row with missing review_text, as it's the core of the dataset
df = df.dropna(subset=['review_text']).reset_index(drop=True)

#fill missing country with 'Unknown', convert to string, strip whitespace, and convert to uppercase
df['country'] = df['country'].fillna('Unknown')
df['country'] = df['country'].astype(str).str.strip().str.upper()

In [102]:
#remove whitespace from profile_link, convert to string, and check for valid links
df = df.dropna(subset=['profile_link']).reset_index(drop=True)
df['profile_link'] = df['profile_link'].astype(str).str.strip()
mask = ~df['profile_link'].str.startswith('/users/')
print(f"Invalid profile links: {mask.sum()}")

Invalid profile links: 0


In [103]:
#only the numeric value, removing any text and converting to numeric types for review_count and rating
df['review_count'] = pd.to_numeric(df['review_count'].astype(str).str.extract(r'(\d+)')[0], errors='coerce').astype('Int64')
df['rating'] = pd.to_numeric(df['rating'].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce').astype(float)

In [104]:
# Convert date columns to datetime, coercing errors and dropping rows with invalid dates
date_cols = ['review_date', 'date_of_experience']
df = df.dropna(subset=date_cols).reset_index(drop=True)
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce', format='mixed')

In [105]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Remove extra whitespace and newlines inside the text
    text = ' '.join(text.split())
    # Fix common HTML/encoding artifacts if any
    text = text.replace('&amp;', '&').replace('&quot;', '"')
    return text.strip()

df['review_title'] = df['review_title'].apply(clean_text)
df['review_text']  = df['review_text'].apply(clean_text)

In [106]:
print(df.dtypes)

reviewer_name                      object
profile_link                       object
country                            object
review_count                        Int64
review_date           datetime64[ns, UTC]
rating                            float64
review_title                       object
review_text                        object
date_of_experience         datetime64[ns]
dtype: object


In [107]:
print("Shape:", df.shape)

Shape: (20947, 9)


In [108]:
df.isnull().sum()

reviewer_name         0
profile_link          0
country               0
review_count          0
review_date           0
rating                0
review_title          0
review_text           0
date_of_experience    0
dtype: int64

In [109]:
# Save as clean CSV (proper quoting so you never have this problem again)
df.to_csv('Amazon_Reviews_CLEANED.csv', 
          index=False, 
          encoding='utf-8')

print("✅ Cleaned dataset saved as Amazon_Reviews_CLEANED.csv")
print("Final shape:", df.shape)

✅ Cleaned dataset saved as Amazon_Reviews_CLEANED.csv
Final shape: (20947, 9)
